# 연습문제 3-5, 3-8 풀이: 은닉층의 구조(층과 뉴런의 수, 배치 순서)

3-2절 연습문제 3-5와 3-8[도전]의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch03/03-02_example.ipynb`를 참고한다.

In [1]:
# torch 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim

## 사전 준비

은닉층의 뉴런 수를 리스트로 받아 [코드 3-1]과 같은 구조의 모델을 만드는 모델 클래스(`DeepMLP`)를 구현해 사용한다.
은닉층마다 `hidden_layer0`, `hidden_layer1`, ... 이름의 계층을 만들려고 파이썬 내장 함수 `setattr()`와 `getattr()`를 쓴다.
`setattr(self, '이름', 값)`은 `self.이름 = 값`과 같고, `getattr(self, '이름')`은 `self.이름`과 같다.
0.5를 문턱값으로 네 샘플을 모두 바르게 분류하면 성공으로 보고 시드를 바꿔 여러 번 학습한다.
학습 환경은 본문과 같다: Adam, 학습률 0.01, 평균제곱오차 손실, 1,000 에포크.

In [2]:
class DeepMLP(nn.Module):                           # [코드 3-1]의 MLP를 일반화한 클래스
    def __init__(self, hidden_sizes):
        super().__init__()
        self.num_hidden = len(hidden_sizes)
        in_features = 2
        # 은닉층이 몇 개인지는 hidden_sizes의 길이가 정한다. 그래서 계층을
        # 담을 변수 이름을 코드에 미리 적어 둘 수 없고, 반복문 안에서 이름을
        # 만들어 붙여야 한다. 이때 쓰는 것이 setattr()다.
        #
        # setattr(객체, '이름', 값)은 `객체.이름 = 값`과 똑같이 동작한다.
        # 다른 점은 이름을 문자열로 줄 수 있어서, 실행 중에 이름을 만들 수
        # 있다는 것이다. hidden_sizes = [10, 8]이면 아래 반복문은 다음 두
        # 줄을 쓴 것과 같다.
        #     self.hidden_layer0 = nn.Linear(2, 10)
        #     self.hidden_layer1 = nn.Linear(10, 8)
        #
        # - enumerate(): (번호, 값) 쌍을 차례로 꺼낸다. 번호는 0부터 시작한다.
        # - f'hidden_layer{i}': 문자열 앞의 f는 { } 안의 변수 값을 문자열에
        #   끼워 넣겠다는 표시다. i가 0이면 'hidden_layer0', 1이면
        #   'hidden_layer1'이 된다.
        # - setattr()로 붙인 계층도 `self.이름 = 계층`으로 붙인 것과 똑같이
        #   파이토치가 모델의 일부로 등록하고 파라미터를 추적한다. 계층을 그냥
        #   파이썬 리스트에 담아 두기만 하면 등록되지 않아 학습되지 않는다.
        for i, out_features in enumerate(hidden_sizes):
            setattr(self, f'hidden_layer{i}', nn.Linear(in_features, out_features))
            in_features = out_features              # 이 층의 출력 수가 다음 층의 입력 수
        self.output_layer = nn.Linear(in_features, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        for i in range(self.num_hidden):            # 은닉층: 선형 계층 + 시그모이드
            # getattr(객체, '이름')은 `객체.이름`과 같다. setattr()로 붙여 둔
            # 계층을 이름(문자열)으로 다시 꺼내고, 바로 뒤의 (x)로 그 계층에
            # x를 넣어 계산한다.
            x = self.activation(getattr(self, f'hidden_layer{i}')(x))
        return self.activation(self.output_layer(x))

In [3]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])    # (4, 2): 입력
Y_true = torch.tensor([[0.], [1.], [1.], [0.]])               # (4, 1): XOR 정답

def train_quiet(model, epochs):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    for _ in range(epochs):
        loss = criterion(model(X), Y_true)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        ok = torch.equal((model(X) >= 0.5).float(), Y_true)   # 네 샘플을 모두 맞히면 성공
    return loss.item(), ok

In [4]:
def compare(configs, seeds, epochs=1000):
    print('구조     파라미터    성공 평균 손실')
    for name, sizes in configs.items():
        results = []
        for seed in seeds:
            torch.manual_seed(seed)                 # 시드마다 다른 초깃값
            results.append(train_quiet(DeepMLP(sizes), epochs))
        n_params = sum(p.numel() for p in DeepMLP(sizes).parameters())
        success = sum(ok for _, ok in results)
        mean_loss = sum(loss for loss, _ in results) / len(results)
        ratio = f'{success}/{len(results)}'
        print(f'{name:8} {n_params:8d} {ratio:>7} {mean_loss:9.4f}')

## 연습문제 3-5

> [코드 3-1]에서 정의한 모델 클래스에서 다음과 같이 은닉층의 수와 뉴런의 수를 바꿔 가며 모델의 학습 결과를 확인해 보자.
> - 뉴런의 수가 세 개인 은닉층 두 개를 포함한 다층 퍼셉트론 모델
> - 뉴런의 수가 세 개인 은닉층 여섯 개를 포함한 다층 퍼셉트론 모델
> - 뉴런의 수가 열 개인 은닉층 하나를 가진 다층 퍼셉트론 모델
> - 뉴런의 수가 각각 세 개, 두 개인 은닉층 두 개를 포함한 다층 퍼셉트론 모델

**네 구조를 시드 10개씩 학습해 비교하기**
- 리스트 `[3, 3]`은 뉴런이 3개, 3개인 은닉층 두 개를 뜻함
- 비교의 기준으로 본문 [코드 3-1]의 구조(뉴런 2개짜리 은닉층 하나)도 함께 학습
- 성공 횟수, 파라미터 수, 평균 손실을 같이 확인

In [5]:
CONFIGS = {
    '2': [2],                               # 본문 [코드 3-1] 모델과 같은 구조
    '3, 3': [3, 3],
    '3 x 6': [3] * 6,
    '10': [10],
    '3, 2': [3, 2],
}
compare(CONFIGS, range(1, 11))

구조     파라미터    성공 평균 손실
2               9    5/10    0.0696
3, 3           25    8/10    0.0298
3 x 6          73    5/10    0.0794
10             41   10/10    0.0004
3, 2           20    7/10    0.0469


**결과 분석: 뉴런이 10개인 은닉층 하나가 10번 모두 성공**
- 10개짜리 은닉층 하나: 10번 중 10번 성공, 평균 손실 0.0004로 가장 안정적
- 3개, 3개: 8번 성공으로 본문의 모델(5번)보다 나음
- 3개, 2개: 7번 성공
- 은닉층 여섯 개: 파라미터가 73개로 가장 많은데도 5번 성공으로 본문의 모델과 같고 평균 손실도 가장 큼
- 깊게 쌓는다고 좋아지지 않고, 한 층의 뉴런을 늘리는 쪽이 더 확실함

**은닉층이 많은 모델이 나빠지는 이유 확인하기**
- 본문에서 언급한 기울기 소실이 원인일 가능성: 시그모이드는 기울기를 줄이면서 전달해 앞쪽 계층일수록 기울기가 작아짐
- 학습 전에 한 번 역전파해서 첫 번째 계층과 출력 계층의 기울기 크기(가중치 기울기의 노름)를 비교
- 텐서의 노름은 텐서의 모든 요솟값을 제곱해 더한 뒤 제곱근을 취한 값이고, 요솟값들의 절대적인 크기로 텐서의 전반적인 강도를 알려 주는 의미를 가진다. 기울기의 노름이 0에 가까우면 가중치가 거의 바뀌지 않는다. (본문 8장에서 추가 설명함)

In [6]:
def first_last_gradient(sizes, seeds=range(1, 11)):
    first = last = 0.
    for seed in seeds:
        torch.manual_seed(seed)
        model = DeepMLP(sizes)
        nn.MSELoss()(model(X), Y_true).backward()          # 역전파 한 번
        first += model.hidden_layer0.weight.grad.norm().item() / len(seeds)
        last += model.output_layer.weight.grad.norm().item() / len(seeds)
    return first, last

for name, sizes in {'2': [2], '3, 3': [3, 3], '3 x 6': [3] * 6}.items():
    first, last = first_last_gradient(sizes)
    ratio = first / last                                   # 첫 계층 / 출력 계층
    print(f'은닉층 {name:6} 첫 계층 {first:.1e}  출력 계층 {last:.1e}  비율 {ratio:.1e}')

은닉층 2      첫 계층 3.7e-03  출력 계층 2.6e-02  비율 1.4e-01
은닉층 3, 3   첫 계층 5.6e-04  출력 계층 4.3e-02  비율 1.3e-02
은닉층 3 x 6  첫 계층 8.5e-08  출력 계층 2.7e-02  비율 3.2e-06


**결과 확인: 은닉층이 많을수록 첫 계층의 기울기가 급격히 작아짐**
- 은닉층이 1개이면 첫 계층의 기울기가 출력 계층의 약 0.14배
- 은닉층이 2개이면 약 0.013배, 6개이면 약 0.0000032배로 거의 0
- 기울기가 이렇게 작으면 앞쪽 계층은 학습 속도가 아주 느려, 1,000 에포크로는 부족
- 기울기 소실은 다음 절에서 ReLU 활성화 함수를 쓰는 동기가 됨

**직접 해 보기: 은닉층 여섯 개 구조의 에포크를 10배로 늘리면?**
- 같은 `[3] * 6` 구조를 1,000 에포크 대신 10,000 에포크로 시드 10개 학습
- 기울기가 작아서 느릴 뿐이라면 에포크를 늘렸을 때 성공이 늘어야 함

In [7]:
compare({'3 x 6': [3] * 6}, range(1, 11), epochs=10000)

구조     파라미터    성공 평균 손실
3 x 6          73    7/10    0.0500


**결과 확인: 에포크를 10배로 늘리면 성공이 5번에서 7번으로 늘어남**
- 1,000 에포크일 때 5번이던 성공이 10,000 에포크에서 7번으로 증가
- 기울기가 작아 느린 학습을 에포크 수를 늘려서 보완할 수는 있지만, 여전히 깊이가 얕고 파라미터 수가 더 작은 다른 모델에 미치지 못함
- 시그모이드로 깊게 쌓는 구조로는 한계가 있음

## 연습문제 3-8 [도전 문제]

> 본문의 은닉층 역할 설명을 바탕으로, 두 개의 은닉층을 포함한 다층 퍼셉트론에서 첫 번째와 두 번째 은닉층 뉴런의 수가
> 각각 세 개, 두 개인 모델과 두 개, 세 개인 모델의 차이를 유추해 보자.
>
> 힌트: 각 은닉층이 만드는 결정 경계의 수와 은닉 공간의 차원 수에 주목해 보자.

**추론: 은닉층의 뉴런 수는 결정 경계의 수이자 다음 공간의 차원 수**
- 은닉층의 뉴런 하나가 결정 경계 하나를 만들고, 뉴런 수만큼의 출력이 다음 계층이 보는 은닉 공간의 차원이 됨
- 두 구조를 이 기준으로 정리

**추론 정리**
- 3 -> 2 모델: 첫 은닉층이 경계 3개를 그어 데이터를 3차원 은닉 공간으로 펼친 뒤, 두 번째 은닉층이 그 안에서 경계 2개로 골라냄
- 2 -> 3 모델: 첫 은닉층이 경계 2개로 데이터를 2차원 은닉 공간으로 좁힌 뒤, 두 번째 은닉층이 그 평면에서 경계 3개를 그음
- 첫 번째 은닉층이 만든 차원 수가 뒤에 오는 계층이 쓸 수 있는 정보의 상한: 2차원으로 좁혀진 뒤에는 경계를 아무리 더 그어도 겹쳐진 정보를 되살릴 수 없음
- 따라서 3 -> 2 모델이 더 안정적일 것이라고 유추

**실험으로 확인하기**
- XOR 데이터로 두 구조를 시드 100개씩 학습해 성공 횟수를 비교
- 10회의 결과(시드를 달리 함)로는 우연한 차이와 구분하기 어려워 100회의 결과로 늘려 비교

In [8]:
SHAPES = {'3 -> 2': [3, 2], '2 -> 3': [2, 3]}
compare(SHAPES, range(1, 101))

구조     파라미터    성공 평균 손실
3 -> 2         20  61/100    0.0593
2 -> 3         19  45/100    0.0796


**결과 분석: 3 -> 2의 성공 비율이 더 높지만 큰 차이는 아님**
- 100번 중 3 -> 2 구조가 61번, 2 -> 3 구조가 45번 성공: 유추한 방향과 같음
- 16회의 차이가 발생하지만 완전히 갈리지는 않음
- XOR 데이터는 점이 네 개뿐이라 2차원으로 좁혀도 잃는 정보가 거의 없기 때문: 점이 많고 복잡한 데이터일수록 차이가 커질 것
- 그래서 앞쪽 은닉층은 넓게 두고 뒤로 갈수록 좁히는 구조를 많이 씀